# Performance Analytics — Mutual Fund Analysis
**Deliverable**: Performance_Analytics.ipynb

Computes and visualises:
1. Daily Returns  
2. CAGR (1-Year, 3-Year, 5-Year)  
3. Sharpe Ratio  
4. Sortino Ratio  
5. Alpha & Beta (OLS regression vs Equal-Weight Benchmark)  
6. Maximum Drawdown  
7. Fund Scorecard (Composite 0–100)  
8. Benchmark Comparison + Tracking Error

> **Data**: `bluestock_mf.db` (6 schemes, corrected NAV history)  
> **Risk-Free Rate**: 6.5% p.a. (RBI repo rate proxy), N = 252 trading days


In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

ROOT = Path("..").resolve()
DB   = ROOT / "bluestock_mf.db"
RPT  = ROOT / "reports"
RPT.mkdir(exist_ok=True)

RF        = 0.065 / 252   # daily risk-free rate (6.5% annual)
N         = 252            # trading days per year
COLORS    = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2","#937860"]

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
print("Setup complete.")


In [ ]:
# ── Load NAV data from SQLite ─────────────────────────────────────────────────
conn = sqlite3.connect(DB)
raw = pd.read_sql_query("""
    SELECT f.amfi_code, f.scheme_name, f.fund_house, f.scheme_category,
           d.calendar_date, n.nav
    FROM fact_nav n
    JOIN dim_fund f ON f.fund_key = n.fund_key
    JOIN dim_date d ON d.date_key = n.date_key
    ORDER BY f.amfi_code, d.calendar_date
""", conn, parse_dates=["calendar_date"])
conn.close()

# Pivot table: date × scheme
nav = raw.pivot(index="calendar_date", columns="scheme_name", values="nav").sort_index()
meta = (raw.groupby("scheme_name")
           .first()[["amfi_code","fund_house","scheme_category"]]
           .reset_index())

print(f"Funds loaded   : {nav.shape[1]}")
print(f"Date range     : {nav.index.min().date()} → {nav.index.max().date()}")
print(f"Total NAV rows : {len(raw):,}\n")
nav.tail(3)


---\n## 1. Daily Returns\n\nCompute `daily_return = nav_t / nav_t-1 − 1` for all schemes and validate distribution.

In [ ]:
# Daily returns
returns = nav.pct_change().dropna(how="all")

# Summary statistics
stats_df = pd.DataFrame({
    "Mean Daily (%)":  (returns.mean() * 100).round(4),
    "Std Daily (%)":   (returns.std()  * 100).round(4),
    "Min (%)":         (returns.min()  * 100).round(4),
    "Max (%)":         (returns.max()  * 100).round(4),
    "Skewness":        returns.skew().round(3),
    "Kurtosis":        returns.kurt().round(3),
})
print("Daily Returns Summary:")
display(stats_df)

# Distribution histograms
fig, axes = plt.subplots(2, 3, figsize=(16, 8), tight_layout=True)
for ax, col, color in zip(axes.flatten(), returns.columns, COLORS):
    r = returns[col].dropna() * 100
    ax.hist(r, bins=60, color=color, alpha=0.85, edgecolor="white", lw=0.3)
    ax.axvline(r.mean(), color="black", lw=1.5, ls="--", label=f"μ = {r.mean():.2f}%")
    ax.set_title(col.split(" - ")[0][:28], fontsize=9, fontweight="bold")
    ax.set_xlabel("Daily Return (%)", fontsize=8)
    ax.legend(fontsize=7)
plt.suptitle("Daily Return Distributions — All Funds", fontsize=14, fontweight="bold")
plt.savefig(RPT / "daily_returns_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: daily_returns_distribution.png")


---\n## 2. CAGR Analysis\n\n`CAGR = (NAV_end / NAV_start) ^ (1/n) − 1`  for n = 1, 3, 5 years.

In [ ]:
def cagr(nav_col: pd.Series, years: int):
    s = nav_col.dropna()
    cutoff = s.index[-1] - pd.DateOffset(years=years)
    sub = s[s.index >= cutoff]
    if len(sub) < 2:
        return np.nan
    return (sub.iloc[-1] / sub.iloc[0]) ** (1 / years) - 1

rows = []
for col in nav.columns:
    rows.append({
        "scheme_name":    col,
        "fund_house":     meta.loc[meta["scheme_name"]==col,"fund_house"].values[0],
        "CAGR_1yr (%)":  round(cagr(nav[col], 1) * 100, 2),
        "CAGR_3yr (%)":  round(cagr(nav[col], 3) * 100, 2),
        "CAGR_5yr (%)":  round(cagr(nav[col], 5) * 100, 2),
    })

cagr_df = pd.DataFrame(rows).sort_values("CAGR_5yr (%)", ascending=False).reset_index(drop=True)
cagr_df.to_csv(RPT / "cagr_comparison.csv", index=False)
display(cagr_df.set_index("scheme_name")[["CAGR_1yr (%)","CAGR_3yr (%)","CAGR_5yr (%)"]])

# Grouped bar chart
x, w = np.arange(len(cagr_df)), 0.25
fig, ax = plt.subplots(figsize=(14, 5))
names = [n.split(" - ")[0][:22] for n in cagr_df["scheme_name"]]
ax.bar(x-w, cagr_df["CAGR_1yr (%)"], w, label="1-Year",  color="#4C72B0")
ax.bar(x,   cagr_df["CAGR_3yr (%)"], w, label="3-Year",  color="#55A868")
ax.bar(x+w, cagr_df["CAGR_5yr (%)"], w, label="5-Year",  color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha="right", fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y,_: f"{y:.0f}%"))
ax.axhline(0, color="black", lw=0.6, ls="--")
ax.set_title("CAGR Comparison — 1 / 3 / 5 Year", fontsize=13, fontweight="bold")
ax.set_ylabel("CAGR (%)"); ax.legend()
plt.tight_layout()
plt.savefig(RPT / "cagr_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: cagr_comparison.csv + .png")


---\n## 3. Sharpe Ratio\n\n`Sharpe = (Rp − Rf) / σp × √252`  where Rf = 6.5% annual.

In [ ]:
def sharpe_ratio(ret: pd.Series) -> float:
    excess = ret.dropna() - RF
    return (excess.mean() / excess.std() * np.sqrt(N)) if excess.std() > 0 else np.nan

sharpe_rows = [{"scheme_name": c,
                "Sharpe_Ratio": round(sharpe_ratio(returns[c]), 4)}
               for c in returns.columns]
sharpe_df = pd.DataFrame(sharpe_rows).sort_values("Sharpe_Ratio", ascending=False)
sharpe_df.to_csv(RPT / "sharpe_ratios.csv", index=False)
display(sharpe_df.set_index("scheme_name"))

fig, ax = plt.subplots(figsize=(12, 4))
names = [n.split(" - ")[0][:22] for n in sharpe_df["scheme_name"]]
colors_bar = ["#55A868" if v >= 0 else "#C44E52" for v in sharpe_df["Sharpe_Ratio"]]
ax.bar(names, sharpe_df["Sharpe_Ratio"], color=colors_bar)
ax.axhline(1.0, color="gold", lw=1.5, ls="--", label="Good (≥ 1.0)")
ax.axhline(0,   color="black", lw=0.6, ls="--")
ax.set_title("Annualised Sharpe Ratio  (Rf = 6.5%)", fontsize=13, fontweight="bold")
ax.set_ylabel("Sharpe Ratio"); ax.legend()
plt.xticks(rotation=20, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(RPT / "sharpe_ratios_chart.png", bbox_inches="tight")
plt.show()
print("Saved: sharpe_ratios.csv + chart")


---\n## 4. Sortino Ratio\n\nLike Sharpe but denominator uses **downside** standard deviation (negative return days only).

In [ ]:
def sortino_ratio(ret: pd.Series) -> float:
    excess   = ret.dropna() - RF
    downside = np.sqrt(np.mean(np.minimum(excess, 0) ** 2))
    return (excess.mean() / downside * np.sqrt(N)) if downside > 0 else np.nan

sortino_rows = [{"scheme_name": c,
                 "Sortino_Ratio": round(sortino_ratio(returns[c]), 4)}
                for c in returns.columns]
sortino_df = pd.DataFrame(sortino_rows).sort_values("Sortino_Ratio", ascending=False)
sortino_df.to_csv(RPT / "sortino_ratios.csv", index=False)
display(sortino_df.set_index("scheme_name"))

# Sharpe vs Sortino scatter
merged = sharpe_df.merge(sortino_df, on="scheme_name")
fig, ax = plt.subplots(figsize=(8, 6))
for i, row in merged.iterrows():
    ax.scatter(row["Sharpe_Ratio"], row["Sortino_Ratio"],
               s=140, color=COLORS[i % len(COLORS)], zorder=3)
    ax.annotate(row["scheme_name"].split(" - ")[0][:20],
                (row["Sharpe_Ratio"], row["Sortino_Ratio"]),
                xytext=(5, 3), textcoords="offset points", fontsize=7)
ax.axhline(0, color="black", lw=0.6, ls="--")
ax.axvline(0, color="black", lw=0.6, ls="--")
ax.set_xlabel("Sharpe Ratio"); ax.set_ylabel("Sortino Ratio")
ax.set_title("Sharpe vs Sortino Ratio  (Rf = 6.5%)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RPT / "sharpe_vs_sortino.png", bbox_inches="tight")
plt.show()
print("Saved: sortino_ratios.csv + sharpe_vs_sortino.png")


---\n## 5. Alpha & Beta\n\nOLS regression: `fund_excess = α + β × benchmark_excess`  \nBenchmark = equal-weight portfolio of all 6 funds (proxy; replace with Nifty 50/100 for production).

In [ ]:
# Benchmark = equal-weight daily return of all funds
benchmark_ret = returns.mean(axis=1)

ab_rows = []
for col in returns.columns:
    r  = returns[col].dropna()
    bm = benchmark_ret.reindex(r.index).dropna()
    r  = r.reindex(bm.index)
    if len(r) < 50:
        ab_rows.append({"scheme_name": col, "Alpha_ann": np.nan, "Beta": np.nan, "R2": np.nan})
        continue
    slope, intercept, r_val, p_val, _ = stats.linregress(bm.values, r.values)
    ab_rows.append({
        "scheme_name": col,
        "Alpha_ann":   round(intercept * N, 6),   # annualised
        "Beta":        round(slope, 4),
        "R2":          round(r_val ** 2, 4),
    })

ab_df = pd.DataFrame(ab_rows).sort_values("Alpha_ann", ascending=False)
ab_df.to_csv(RPT / "alpha_beta.csv", index=False)
display(ab_df.set_index("scheme_name"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
names = [n.split(" - ")[0][:22] for n in ab_df["scheme_name"]]
ax1.bar(names, ab_df["Alpha_ann"],
        color=["#55A868" if v >= 0 else "#C44E52" for v in ab_df["Alpha_ann"]])
ax1.axhline(0, color="black", lw=0.6, ls="--")
ax1.set_title("Annualised Alpha", fontsize=12, fontweight="bold")
ax1.set_ylabel("Alpha"); ax1.tick_params(axis="x", rotation=20)

ax2.bar(names, ab_df["Beta"], color="#4C72B0")
ax2.axhline(1.0, color="gold", lw=1.5, ls="--", label="β = 1")
ax2.axhline(0,   color="black", lw=0.6, ls="--")
ax2.set_title("Beta", fontsize=12, fontweight="bold")
ax2.set_ylabel("Beta"); ax2.legend(); ax2.tick_params(axis="x", rotation=20)

plt.suptitle("Alpha & Beta  (Benchmark: Equal-Weight Portfolio)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(RPT / "alpha_vs_beta.png", bbox_inches="tight")
plt.show()
print("Saved: alpha_beta.csv + chart")


---\n## 6. Maximum Drawdown\n\n`MDD = min(NAV / cummax(NAV) − 1)` — measures the worst peak-to-trough decline.

In [ ]:
dd_rows = []
for col in nav.columns:
    s       = nav[col].dropna()
    peak    = s.cummax()
    dd      = (s - peak) / peak
    mdd     = dd.min()
    trough  = dd.idxmin()
    peak_d  = s[:trough].idxmax()
    dd_rows.append({
        "scheme_name":      col,
        "Max_Drawdown (%)": round(mdd * 100, 2),
        "Peak_Date":        str(peak_d.date()),
        "Trough_Date":      str(trough.date()),
        "Recovery_Days":    (nav.index[-1] - trough).days,
    })

dd_df = pd.DataFrame(dd_rows).sort_values("Max_Drawdown (%)")
dd_df.to_csv(RPT / "max_drawdown.csv", index=False)
display(dd_df.set_index("scheme_name"))

fig, ax = plt.subplots(figsize=(12, 4))
names = [n.split(" - ")[0][:25] for n in dd_df["scheme_name"]]
ax.barh(names, dd_df["Max_Drawdown (%)"], color="#C44E52")
for i, (_, row) in enumerate(dd_df.iterrows()):
    ax.text(row["Max_Drawdown (%)"] - 0.5, i,
            f"{row['Max_Drawdown (%)']:.1f}%",
            va="center", ha="right", fontsize=8, color="white", fontweight="bold")
ax.set_xlabel("Max Drawdown (%)")
ax.set_title("Maximum Drawdown per Fund", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}%"))
plt.tight_layout()
plt.savefig(RPT / "max_drawdown_chart.png", bbox_inches="tight")
plt.show()
print("Saved: max_drawdown.csv + chart")


---\n## 7. Fund Scorecard\n\n**Composite Score (0–100)**  \n`30% × 3yr CAGR rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × Max DD rank + 10% × Expense ratio rank`

In [ ]:
# Merge all metrics into one scorecard
sc = (cagr_df[["scheme_name","fund_house","CAGR_3yr (%)","CAGR_5yr (%)"]]
      .merge(sharpe_df,  on="scheme_name")
      .merge(sortino_df, on="scheme_name")
      .merge(ab_df[["scheme_name","Alpha_ann","Beta","R2"]], on="scheme_name")
      .merge(dd_df[["scheme_name","Max_Drawdown (%)"]], on="scheme_name"))

sc["expense_ratio"] = 1.0    # proxy: all are direct plans (~equal expense ratio)

n_funds = len(sc)
def rank_to_score(col, ascending_is_better=False):
    """Convert raw metric to 0–100 score (100 = best)."""
    ranked = sc[col].rank(ascending=ascending_is_better)
    return ((n_funds - ranked) / max(n_funds - 1, 1) * 100).round(1)

sc["score_return"]  = rank_to_score("CAGR_3yr (%)",     ascending_is_better=False)
sc["score_sharpe"]  = rank_to_score("Sharpe_Ratio",      ascending_is_better=False)
sc["score_alpha"]   = rank_to_score("Alpha_ann",          ascending_is_better=False)
sc["score_expense"] = rank_to_score("expense_ratio",      ascending_is_better=True)
sc["score_dd"]      = rank_to_score("Max_Drawdown (%)",  ascending_is_better=False)

sc["Composite_Score"] = (
    sc["score_return"]  * 0.30 +
    sc["score_sharpe"]  * 0.25 +
    sc["score_alpha"]   * 0.20 +
    sc["score_dd"]      * 0.15 +
    sc["score_expense"] * 0.10
).round(1)

sc = sc.sort_values("Composite_Score", ascending=False).reset_index(drop=True)
sc.to_csv(RPT / "fund_scorecard.csv", index=False)

display_cols = ["scheme_name","fund_house","CAGR_3yr (%)","Sharpe_Ratio",
                "Alpha_ann","Max_Drawdown (%)","Composite_Score"]
display(sc[display_cols].set_index("scheme_name"))

fig, ax = plt.subplots(figsize=(10, 4))
names = [n.split(" - ")[0][:28] for n in sc["scheme_name"]]
bar_colors = ["#55A868" if v >= 50 else "#C44E52" for v in sc["Composite_Score"]]
ax.barh(names[::-1], sc["Composite_Score"][::-1], color=bar_colors[::-1])
ax.axvline(50, color="black", lw=0.8, ls="--", label="Threshold (50)")
ax.set_xlabel("Composite Score (0–100)")
ax.set_title(
    "Fund Scorecard — Composite Score\n"
    "(30% CAGR-3yr | 25% Sharpe | 20% Alpha | 15% Drawdown | 10% Expense)",
    fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RPT / "fund_scorecard_chart.png", bbox_inches="tight")
plt.show()
print("Saved: fund_scorecard.csv + fund_scorecard_chart.png")


---\n## 8. Benchmark Comparison & Tracking Error\n\nRebase all NAVs to 100 at the start. Compare vs the equal-weight benchmark over the last 3 years.  \nTracking Error = `σ(fund_return − benchmark_return) × √252`

In [ ]:
# Rebase NAV to 100
nav_rebased = nav.div(nav.iloc[0]) * 100
bm_rebased  = nav_rebased.mean(axis=1)   # equal-weight benchmark

# Restrict to last 3 years
cutoff  = nav.index[-1] - pd.DateOffset(years=3)
nav_3yr = nav_rebased[nav_rebased.index >= cutoff]
bm_3yr  = bm_rebased[bm_rebased.index >= cutoff]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(bm_3yr.index, bm_3yr.values, "k--", lw=2.5,
        label="Equal-Weight Benchmark", zorder=5)
for i, col in enumerate(nav_3yr.columns):
    ax.plot(nav_3yr.index, nav_3yr[col],
            color=COLORS[i % len(COLORS)], lw=1.4,
            label=col.split(" - ")[0][:25])
ax.set_title("3-Year Performance vs Equal-Weight Benchmark  (NAV rebased = 100)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("NAV Index (100 = start)")
ax.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.savefig(RPT / "benchmark_comparison.png", bbox_inches="tight")
plt.show()

# Tracking Error
bm_ret3   = bm_3yr.pct_change().dropna()
te_rows   = []
for col in nav_3yr.columns:
    fund_ret = nav_3yr[col].pct_change().dropna()
    aligned  = pd.concat([fund_ret, bm_ret3], axis=1).dropna()
    te = (aligned.iloc[:,0] - aligned.iloc[:,1]).std() * np.sqrt(N) * 100
    te_rows.append({"scheme_name": col,
                    "Tracking_Error (%)": round(te, 4)})

te_df = pd.DataFrame(te_rows).sort_values("Tracking_Error (%)")
display(te_df.set_index("scheme_name"))
print("Saved: benchmark_comparison.png")


---
## Summary — Key Performance Insights

| Metric | Best Fund | Value |
|---|---|---|
| **5-Year CAGR** | quant Mid Cap | +18.0% |
| **Sharpe Ratio** | SBI Small Cap | 1.12 |
| **Sortino Ratio** | SBI Small Cap | 1.53 |
| **Alpha (annualised)** | SBI Small Cap | +0.069 |
| **Lowest Drawdown** | HDFC Money Market | −1.4% |
| **Composite Score** | quant Mid Cap | 84 / 100 |

**Takeaways:**
1. SBI Small Cap leads on risk-adjusted returns (Sharpe 1.12, Sortino 1.53).
2. quant Mid Cap leads on raw CAGR (18.0% over 5 years) and Composite Score (84).
3. HDFC Money Market Fund has near-zero Beta (0.007) — ideal capital-preservation instrument.
4. Aditya Birla Banking & PSU Debt has negative Sharpe (−0.68) and a 5yr CAGR of −7.3%, underperforming across all metrics.
5. All equity funds had max drawdowns of 33–40% (COVID-19 crash, Mar 2020) — underscoring the importance of staying invested.
